In [1]:
import sys

omix_path = "/workspace/1AlgoG3/python_packages/multiomix"
fileverse_path = "/workspace/1AlgoG3/python_packages/fileverse17"
statomix_path = "/workspace/1AlgoG3/projects/germinal_centre/statomix"

sys.path.append(omix_path)
sys.path.append(statomix_path)
sys.path.append(fileverse_path)

In [2]:
import pandas as pd
from pathlib import Path
from dataclasses import dataclass

from fileverse.logger import Logger
from fileverse.formats.yaml import BaseYAML
from fileverse.formats.zarr import BaseZARR
from fileverse.formats.excel import BaseExcel

from statomix.project.project import Project
from statomix.pipelines.cleaner.col.col_semantic_rules import DataTypes

In [3]:
path_N0 = "/workspace/1AlgoG3/projects/germinal_centre/gc_w_clinical_csv/N0_Updated.csv"
path_OCAT = "/workspace/1AlgoG3/projects/germinal_centre/gc_w_clinical_csv/OCAT/patient_wise_stats_with_clinical_data_source_updated.csv"
path_Priyanka = "/workspace/1AlgoG3/projects/germinal_centre/gc_w_clinical_csv/Priyanka_Thesis/patient_wise_stats_with_clinical_data-UPDATED.xlsx"

df_N0 = pd.read_csv(path_N0)
df_OCAT = pd.read_csv(path_OCAT)
df_Priyanka = pd.read_excel(path_Priyanka)

common_pids = set(df_N0['Patient ID']).intersection(set(df_Priyanka['Patient ID']))

df_Priyanka = df_Priyanka[~df_Priyanka["Patient ID"].isin(common_pids)]

In [4]:
project_name="Germinal Center (Statomix Trial 2)"

In [5]:
project = Project(project_name=project_name)

# df = df_Priyanka
# dataset_name = "Priyanka"

# project.add_dataset(df=df, dataset_name=dataset_name)

INFO (2026-08-03 09:55:33)
Discovered and loaded existing dataset: 'N0'
INFO (2026-08-03 09:55:33)
Discovered and loaded existing dataset: 'OCAT'
INFO (2026-08-03 09:55:33)
Discovered and loaded existing dataset: 'Priyanka'


# Cleaner Pipeline

In [6]:
version = None
config_version = None

In [7]:
dataset = project.datasets['OCAT']

In [8]:
cleaner = dataset.cleaner

In [9]:
cleaner.create_cat_meta_edit_schema(version=version, config_version=config_version, create)

SyntaxError: positional argument follows keyword argument (1107429208.py, line 1)

# Summary

In [ ]:
from tqdm.auto import tqdm
from collections import defaultdict

In [ ]:
def get_num_filtered_df_dict(project, num_col_names_df):

    filtered_df_dict = {}
    for _, row in num_col_names_df.iterrows():
        dataset = project.datasets[row['Dataset']]
        group_analyzer = dataset.analyzer._get_group_analyzer(version=None, config_version=None)
    
        column_name_cols = [col for col in col_names_df.columns if col.startswith("Column Name")]
    
        num_summary_df = group_analyzer.get_num_summary_df()
    
        filtered_df = num_summary_df.filter(items=list(row[column_name_cols]), axis=0)
        filtered_df = filtered_df.round(2).reset_index(names="col_name")
    
        filtered_df_dict[row['Dataset']] = filtered_df
    
    return filtered_df_dict

In [ ]:
def get_cat_filtered_df_dict(project, cat_col_names_df):

    filtered_df_dict = {}
    for _, row in cat_col_names_df.iterrows():
        dataset = project.datasets[row['Dataset']]
        group_analyzer = dataset.analyzer._get_group_analyzer(version=None, config_version=None)
    
        column_name_cols = [col for col in col_names_df.columns if col.startswith("Column Name")]
        
        cat_summary_df = group_analyzer.get_cat_summary_df()

        selected_col_names = (
            row[column_name_cols]
            .dropna()
            .tolist()
        )
    
        filtered_df = cat_summary_df.loc[
            cat_summary_df.index
            .get_level_values("col_name")
            .isin(selected_col_names),
            :
        ]
    
        filtered_df_dict[row['Dataset']] = filtered_df

    return filtered_df_dict

In [ ]:
analysis_config_df = pd.read_excel('analysis_config_version1_curated.xlsx')

In [ ]:
uids = analysis_config_df['UID'].dropna().unique()

In [ ]:
sorted_uids = defaultdict(dict)

for uid in uids:
    uid_df =(analysis_config_df[analysis_config_df['UID']==uid]).copy()

    num_df = uid_df[uid_df['Datatype'] == 'Numerical']
    num_df = num_df.dropna(axis=1)
    
    cat_df = uid_df[uid_df['Datatype'] == 'Categorical']
    cat_df = cat_df.dropna(axis=1)

    if not num_df.empty:
        sorted_uids[uid]['num'] =  num_df

    if not cat_df.empty:
        sorted_uids[uid]['cat'] =  cat_df

In [ ]:
filtered_data = defaultdict(dict)
for uid, v in tqdm(sorted_uids.items()):
    for datatype, col_names_df in v.items():
        if datatype == 'cat':
            filtered_df_dict = get_cat_filtered_df_dict(project=project, cat_col_names_df=col_names_df)
        elif datatype == 'num':
            filtered_df_dict = get_num_filtered_df_dict(project=project, num_col_names_df=col_names_df)

        filtered_data[uid][datatype] =  filtered_df_dict

In [ ]:
from num_table import create_journal_summary_table
from cat_table import create_journal_categorical_table

In [ ]:
tables = defaultdict(dict)
for k , v in filtered_data.items():
    try:
        for datatype, summary_df in v.items():
            if datatype == 'cat':
                table = create_journal_categorical_table(summaries=summary_df, title=k)
            elif datatype == "num":
                table = create_journal_summary_table(summaries=summary_df, title=k)
            else:
                raise ValueError(f"Unknown Datatype")
            tables[k][datatype] = table
    except Exception as e:
        print(e)
        print(f"{k}")

In [ ]:
tables['Computational']['num']